# Cookie Cats A/B Test: Gate at Level 30 vs Level 40

**Question:** Does moving the first progress gate from level 30 (`gate_30`, control) to level 40 (`gate_40`, treatment) change player retention and engagement?

- **Primary metric:** `retention_7`
- **Secondary metrics:** `retention_1`, `sum_gamerounds`
- **Settings:** α = 0.05, two-sided tests, 95% confidence intervals, intention-to-treat (all randomized players)

## Part 1 – Setup
### 1.1 Configuration
Edit `DATA_PATH` if your file sits somewhere else in Google Drive.

In [ ]:
# ---- Edit this path if needed ----
DATA_PATH = "/content/drive/MyDrive/cookie_cats.csv"

# Experiment settings
CONTROL, TREATMENT = "gate_30", "gate_40"
ALPHA = 0.05          # significance level (two-sided)
CONF_LEVEL = 0.95     # confidence level for all intervals
SEED = 42             # seed for bootstrap and permutation steps
N_BOOT = 5_000        # bootstrap / permutation resamples

### 1.2 Imports and plot style

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import (
    proportions_ztest,
    confint_proportions_2indep,
    proportion_effectsize,
)
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)

# Reproducibility: one seeded generator for all resampling
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# Plot style
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 100,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
})
PALETTE = {CONTROL: "#1f77b4", TREATMENT: "#ff7f0e"}

# Table display
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print(f"pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}")
print(f"Seed = {SEED}, alpha = {ALPHA}, bootstrap resamples = {N_BOOT:,}")

### 1.3 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.4 Load the data

In [ ]:
def load_data(path):
    """Load the CSV; print a clear message and stop if the path is wrong."""
    if not os.path.isfile(path):
        print(f"ERROR: file not found at:\n   {path}")
        print("Check DATA_PATH in cell 1.1 (spelling, folder, and that Drive is mounted).")
        # Help locate the file: list CSVs in the top level of My Drive
        candidates = sorted(glob.glob("/content/drive/MyDrive/*.csv"))
        if candidates:
            print("\nCSV files found in My Drive:")
            for c in candidates[:20]:
                print("   ", c)
        raise FileNotFoundError(path)
    return pd.read_csv(path)

df = load_data(DATA_PATH)

print(f"Loaded {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Quick check against the expected size
EXPECTED_SHAPE = (90_189, 5)
if df.shape == EXPECTED_SHAPE:
    print("Shape matches the expected 90,189 x 5.")
else:
    print(f"Warning: expected {EXPECTED_SHAPE}, got {df.shape}. Confirm this is the right file.")

df.head()

## Part 2 – Data Integrity
### 2.1 Column types and a quality log
Every check below writes one line to a data quality log, printed at the end of the part.

In [ ]:
# Data quality log: one row per check
dq_log = []

def log_check(check, passed, detail):
    """Record a data quality check and print it."""
    status = "PASS" if passed else "FLAG"
    dq_log.append({"check": check, "status": status, "detail": detail})
    print(f"[{status}] {check}: {detail}")

print("Column types as loaded:")
print(df.dtypes.to_string())

### 2.2 Convert retention columns to boolean
`pd.read_csv` usually reads `TRUE`/`FALSE` as booleans already. If not (for example strings or 0/1), this cell converts them and flags any value it cannot map.

In [ ]:
BOOL_MAP = {
    "true": True, "false": False,
    "t": True, "f": False,
    "yes": True, "no": False,
    "1": True, "0": False,
}

def to_bool(series):
    """Convert a column to bool; return (converted, number of unmapped values)."""
    if pd.api.types.is_bool_dtype(series):
        return series, 0
    mapped = series.astype(str).str.strip().str.lower().map(BOOL_MAP)
    n_bad = int(mapped.isna().sum() - series.isna().sum())
    return mapped, n_bad

for col in ["retention_1", "retention_7"]:
    original_dtype = df[col].dtype
    converted, n_bad = to_bool(df[col])
    if n_bad == 0 and converted.notna().all():
        df[col] = converted.astype(bool)
        note = "already boolean" if original_dtype == bool else f"converted from {original_dtype}"
        log_check(f"{col} is boolean", True, note)
    else:
        df[col] = converted  # keep NaN for unmapped values so they show up as missing
        log_check(f"{col} is boolean", False, f"{n_bad:,} values could not be mapped to True/False")

### 2.3 Unique IDs, missing values, duplicates

In [ ]:
# userid uniqueness
n_ids = df["userid"].nunique()
n_dup_ids = len(df) - n_ids
log_check("userid is unique", n_dup_ids == 0,
          f"{n_ids:,} unique IDs across {len(df):,} rows ({n_dup_ids:,} repeated)")

# Missing values per column
missing = df.isna().sum()
n_missing = int(missing.sum())
detail = "no missing values" if n_missing == 0 else \
    ", ".join(f"{c}={m:,}" for c, m in missing.items() if m > 0)
log_check("No missing values", n_missing == 0, detail)

# Fully duplicated rows
n_dup_rows = int(df.duplicated().sum())
log_check("No duplicated rows", n_dup_rows == 0, f"{n_dup_rows:,} fully duplicated rows")

# A repeated userid would mean a player was counted twice (possibly in both groups)
if n_dup_ids > 0:
    dup_ids = df.loc[df["userid"].duplicated(keep=False), "userid"]
    in_both = df[df["userid"].isin(dup_ids)].groupby("userid")["version"].nunique().gt(1).sum()
    print(f"   Note: {in_both:,} repeated IDs appear in both groups. Review before analysis.")

### 2.4 Value checks on `version` and `sum_gamerounds`

In [ ]:
# version must contain exactly the two expected arms
expected_versions = {CONTROL, TREATMENT}
found_versions = set(df["version"].dropna().unique())
unexpected = found_versions - expected_versions
log_check("version has only gate_30 / gate_40", not unexpected and found_versions == expected_versions,
          f"found {sorted(found_versions)}" + (f"; unexpected {sorted(unexpected)}" if unexpected else ""))

# sum_gamerounds must be whole, non-negative numbers
rounds = df["sum_gamerounds"]
is_int = pd.api.types.is_integer_dtype(rounds)
log_check("sum_gamerounds is integer", is_int, f"dtype = {rounds.dtype}")

n_negative = int((rounds < 0).sum())
log_check("sum_gamerounds is non-negative", n_negative == 0, f"{n_negative:,} negative values")

# Zero-round players are kept (they were randomized); just report them
n_zero = int((rounds == 0).sum())
log_check("Zero-round players kept", True,
          f"{n_zero:,} players ({n_zero / len(df):.2%}) played 0 rounds")

# Extreme values: compare the maximum to the next highest value
top_two = rounds.nlargest(2).tolist()
ratio = top_two[0] / top_two[1] if top_two[1] > 0 else np.inf
log_check("No extreme outlier in sum_gamerounds", ratio < 5,
          f"max = {top_two[0]:,}, next highest = {top_two[1]:,} ({ratio:.1f}x); handled in Part 5")

print("\nsum_gamerounds summary:")
print(rounds.describe(percentiles=[0.25, 0.5, 0.75, 0.99, 0.999]).round(1).to_string())

### 2.5 Data quality log

In [ ]:
dq_table = pd.DataFrame(dq_log)
n_flags = (dq_table["status"] == "FLAG").sum()

print(f"Data quality log: {len(dq_table)} checks, {n_flags} flagged\n")
display(dq_table)

if n_flags == 0:
    print("All checks passed. The data is ready for analysis.")
else:
    flagged = ", ".join(dq_table.loc[dq_table["status"] == "FLAG", "check"])
    print(f"Flagged: {flagged}.")
    print("No rows are removed here. The primary analysis keeps every randomized player; "
          "flagged items are handled explicitly in later parts.")

## Part 3 – Experiment Validity
### 3.1 Players per version

In [ ]:
n_by_version = df["version"].value_counts().reindex([CONTROL, TREATMENT])
n_total = int(n_by_version.sum())

counts_table = pd.DataFrame({
    "players": n_by_version,
    "share": (n_by_version / n_total).map("{:.3%}".format),
})
display(counts_table)

gap = int(n_by_version[TREATMENT] - n_by_version[CONTROL])
print(f"{TREATMENT} has {abs(gap):,} {'more' if gap > 0 else 'fewer'} players than {CONTROL} "
      f"({abs(gap) / n_total:.2%} of all players).")

### 3.2 Sample ratio mismatch (SRM) test
Players were meant to be split 50/50. A chi-square goodness-of-fit test checks whether the observed split is consistent with that.
If it is not, the randomization or the logging may be broken, and every comparison that follows carries that caveat.

Many experimentation teams use a stricter threshold (p < 0.001) for SRM alarms, so both thresholds are reported.

In [ ]:
SRM_STRICT_ALPHA = 0.001  # common industry threshold for SRM alarms

def srm_test(observed, expected_ratio=(0.5, 0.5)):
    """Chi-square goodness-of-fit test of group sizes against the planned split."""
    observed = np.asarray(observed, dtype=float)
    expected = observed.sum() * np.asarray(expected_ratio)
    chi2, p = stats.chisquare(observed, f_exp=expected)
    return {"chi2": chi2, "p_value": p, "observed": observed, "expected": expected}

srm_result = srm_test(n_by_version.values)

print("Sample ratio mismatch test (expected 50/50)")
for name, obs, exp in zip(n_by_version.index, srm_result["observed"], srm_result["expected"]):
    print(f"  {name}: observed {obs:,.0f} | expected {exp:,.1f} | diff {obs - exp:+,.1f}")
print(f"  chi-square = {srm_result['chi2']:.3f}, df = 1, p-value = {srm_result['p_value']:.4f}")

# Plain-English interpretation, driven by the numbers
p = srm_result["p_value"]
srm_flag = p < ALPHA
srm_result["flag"] = srm_flag
print("\nInterpretation:")
if p < SRM_STRICT_ALPHA:
    print(f"  p = {p:.4f} is below the strict SRM threshold of {SRM_STRICT_ALPHA}. This is a serious mismatch.")
    print("  The assignment or data collection is likely flawed. Treat all results below with strong caution.")
elif p < ALPHA:
    print(f"  p = {p:.4f} is below alpha = {ALPHA}, so the split differs from 50/50 more than chance alone")
    print(f"  would usually produce. It is above the strict SRM threshold of {SRM_STRICT_ALPHA}, so this is a")
    print("  warning rather than a hard stop. We continue, but record it as a caveat for the conclusion.")
else:
    print(f"  p = {p:.4f} is above alpha = {ALPHA}. The split is consistent with 50/50; no SRM detected.")

### 3.3 Diagnostic: zero-round players by version
Players with 0 rounds never reached either gate, so the gate position could not have affected them.
Their share should be about the same in both groups. A large difference would point to an assignment or logging problem
behind the SRM. We keep these players in every analysis because they were randomized.

In [ ]:
zero_by_version = (
    df.assign(zero_rounds=df["sum_gamerounds"].eq(0))
      .groupby("version")["zero_rounds"]
      .agg(zero_round_players="sum", players="count")
      .reindex([CONTROL, TREATMENT])
)
zero_by_version["share"] = zero_by_version["zero_round_players"] / zero_by_version["players"]
display(zero_by_version.style.format({"zero_round_players": "{:,}", "players": "{:,}", "share": "{:.2%}"}))

n_zero_total = int(zero_by_version["zero_round_players"].sum())
print(f"Total zero-round players: {n_zero_total:,} ({n_zero_total / n_total:.2%} of all players)")

# Are zero-round shares similar across groups?
zero_ct = np.column_stack([
    zero_by_version["zero_round_players"],
    zero_by_version["players"] - zero_by_version["zero_round_players"],
])
chi2_z, p_z, _, _ = stats.chi2_contingency(zero_ct)
diff_pp = (zero_by_version.loc[TREATMENT, "share"] - zero_by_version.loc[CONTROL, "share"]) * 100
print(f"Difference ({TREATMENT} - {CONTROL}): {diff_pp:+.2f} pp | chi-square p-value = {p_z:.4f}")
if p_z < ALPHA:
    print("Interpretation: zero-round shares differ between groups. Because the gate cannot affect these players,")
    print("this suggests the SRM may come from how players were assigned or logged.")
else:
    print("Interpretation: zero-round shares are similar in both groups. This check shows no sign that the")
    print("imbalance comes from a specific type of player, though it cannot rule out other causes.")

### 3.4 Crosstab of `retention_1` vs `retention_7`
Checks how the two retention metrics relate. Some players return on day 7 without returning on day 1, so the metrics are related but not nested.

In [ ]:
ct = pd.crosstab(df["retention_1"], df["retention_7"], margins=True, margins_name="Total")
ct.index.name, ct.columns.name = "retention_1", "retention_7"
print("Counts:")
display(ct)

print("Share of all players:")
display((pd.crosstab(df["retention_1"], df["retention_7"], normalize="all") * 100)
        .round(2).rename_axis(index="retention_1", columns="retention_7"))

# Conditional day-7 retention, by version
cond = (df.groupby(["version", "retention_1"])["retention_7"].mean()
          .unstack().reindex([CONTROL, TREATMENT]))
cond.columns = ["P(ret_7 | not ret_1)", "P(ret_7 | ret_1)"]
print("Day-7 retention conditional on day-1 retention, by version:")
display(cond.style.format("{:.2%}"))

# Interpretation from the numbers
n_r7 = int(df["retention_7"].sum())
n_r7_only = int((df["retention_7"] & ~df["retention_1"]).sum())
r7_given_r1 = df.loc[df["retention_1"], "retention_7"].mean()
r7_given_not_r1 = df.loc[~df["retention_1"], "retention_7"].mean()
print(f"Interpretation: {n_r7:,} players were retained on day 7. Of these, {n_r7_only:,} "
      f"({n_r7_only / n_r7:.1%}) did not return on day 1.")
print(f"Players who returned on day 1 were {r7_given_r1 / r7_given_not_r1:.1f}x as likely to return on day 7 "
      f"({r7_given_r1:.1%} vs {r7_given_not_r1:.1%}). The metrics are strongly linked,")
print("so their test results are not independent. The Holm correction in Part 10 stays valid under this dependence.")

### 3.5 Validity summary

In [ ]:
validity = pd.DataFrame([
    {"check": "Group sizes",
     "result": f"{CONTROL}: {n_by_version[CONTROL]:,} | {TREATMENT}: {n_by_version[TREATMENT]:,}"},
    {"check": "SRM chi-square (50/50)",
     "result": f"chi2 = {srm_result['chi2']:.2f}, p = {srm_result['p_value']:.4f} -> "
               + ("mismatch at alpha 0.05" if srm_result["flag"] else "no mismatch")},
    {"check": "Zero-round share by group",
     "result": f"diff = {diff_pp:+.2f} pp, p = {p_z:.4f}"},
    {"check": "Zero-round players (kept)",
     "result": f"{n_zero_total:,} ({n_zero_total / n_total:.2%})"},
])
display(validity)